# 🇳🇵🇮🇳🇺🇸 Chatterbox Multilingual Fine-tuning

This notebook fine-tunes the Chatterbox TTS model on a combined **Nepali, Maithili, and English** dataset (27GB) using Kaggle T4 x2 GPUs.

**Important**: After running Cell 1 (setup), **restart the Kaggle session** before running Cell 2+.

In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# 1. Setup Environment
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["TRAINING_ENV"] = "kaggle"

# 2. Clone Repository
REPO_URL = "https://github.com/Firojpaudel/chatterbox-nepali.git"
REPO_DIR = "/kaggle/working/chatterbox-nepali"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Updating repository...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# 3. Install Dependencies
print("Installing dependencies...")
subprocess.run(["pip", "uninstall", "-y", "torchcodec"], check=False)

subprocess.run([
    "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)

subprocess.run(["pip", "install", "-e", "."], check=True)

subprocess.run([
    "pip", "install",
    "pandas>=2.0,<2.3",
    "--force-reinstall", "--no-deps"
], check=True)

subprocess.run(["pip", "install", "datasets", "soundfile"], check=True)

subprocess.run([
    "pip", "install",
    "omegaconf", "conformer", "s3tokenizer", "safetensors",
    "pyloudnorm", "wandb", "huggingface-hub", "librosa", "soundfile"
], check=True)
subprocess.run(["pip", "install", "git+https://github.com/resemble-ai/Perth.git@master"], check=True)

print("\n--- Chatterbox Setup Complete ---")

In [ ]:
%%writefile /kaggle/working/chatterbox-nepali/prepare_data.py
import os, sys
from pathlib import Path
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq
from tqdm.auto import tqdm

dataset_repo = 'Firoj112/chatterbox-multilingual-data'
local_dir = Path('finetuning_data')
wavs_dir = local_dir / 'data'
wavs_dir.mkdir(parents=True, exist_ok=True)

# Download manifests + parquet data files
print(f'Downloading dataset from {dataset_repo}...')
snapshot_download(
    repo_id=dataset_repo,
    repo_type='dataset', 
    local_dir=local_dir,
    allow_patterns=['manifests/*.jsonl', 'data/*.parquet'],
    token=os.environ.get('HF_TOKEN')
)

# Extract audio bytes from parquet
parquet_files = sorted((local_dir / 'data').glob('*.parquet'))
print(f'Found {len(parquet_files)} parquet files. Extracting audio...')

for pf in parquet_files:
    table = pq.read_table(pf, columns=['audio'])
    for i in tqdm(range(len(table)), desc=pf.name):
        audio_struct = table['audio'][i].as_py()
        if not audio_struct: continue
        audio_bytes = audio_struct.get('bytes')
        path = audio_struct.get('path', f'audio_{i}.wav')
        if not audio_bytes: continue
        save_path = wavs_dir / Path(path).name
        if not save_path.exists():
            with open(save_path, 'wb') as f:
                f.write(audio_bytes)

print(f'Dataset ready in {wavs_dir}')

In [ ]:
!python /kaggle/working/chatterbox-nepali/prepare_data.py

In [ ]:
# 🚀 START MULTILINGUAL TRAINING (T4 x 2)
# Starting from SCRATCH (no --resume_t3_weights)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  torchrun --nproc_per_node=2 src/chatterbox/train_multilingual.py \
    --manifest finetuning_data/manifests/train_manifest.jsonl \
    --wav_dir finetuning_data/data \
    --push_to_hub Firoj112/chatterbox-multilingual-t3-v1 \
    --batch_size 16 \
    --accum_steps 1 \
    --epochs 50 \
    --lr 5e-5 \
    --save_every 1 \
    --num_workers 2 \
    --distributed \
    --fp16